### Step1: Import Libraries

In [49]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import json
from pprint import pprint
import uuid
import chromadb
#Test Pushover
import requests
import random




load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise Exception("API Key is missing!")
else:
    print("Key is: " + OPENAI_API_KEY[:8])

client = OpenAI(api_key=OPENAI_API_KEY)


Key is: sk-proj-


### Step 2: RAG Preparation

In [ ]:
system_message = """ 

Important: do not make something up. If you don't know the answer, just say I donot know
The only factual information available to you is what is in this system message.
You can't get any more facts about Si Lam from the internet or make something up.
Here is the ONLY factual information about Si Lam

If the question is asked that is not answerable, say I don't know.
"""
f = open('./digital_twin/document_education.txt', 'r', encoding='utf-8')
document_education = f.read()

f = open('./digital_twin/document_overview.txt', 'r', encoding='utf-8')
document_overview = f.read()

f = open('./digital_twin/document_professional.txt', 'r', encoding='utf-8')
document_professional = f.read()

print(document_overview)
print(document_education)
print(document_professional)



In 1999, I Just graduated from Ausbug College

I got my master degree in Software Engineering at University of St. Thomas in 2015
I got my BS at Augsburg College in 1999

I have Microsoft Azure Certified: AI-102, AZ-420, DP-900,DP-700,DP-600,D-100,AZ-400,AZ-104,AZ-204

also I have Amazon Web Service AWS Certified: Administrator and Developer;, 

I am Senior Software Engineer at Red Wing Shoes, working on Front End Javascript, Blazor, and
back end API

You can follow me at my personal website: http://www.serverlessdeveloper.com


My Contact info: silam@hotmail.com 


Here is the list of AWS Certification 

AWS Certified Solution Architect - Associate 
AWS Certified Developer 

Here is the list of Microsoft Azure Certification:

Microsoft Certified Azure AI Engineer Associate (AI-102)
Microsoft Certified Azure Administrator Associate (AZ-104)
Microsoft Certified Developer (AZ-204)
Microsoft Certified DevOps Certified Engineer Expert (AZ-400) 
Microsoft Certified Data Scientist Associate

In [51]:
#Split text into chunks
def split_text_into_chunks(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    
    BOUNDARIES = ["\n\n","\n",". ", "? ", "! "," "]


    def find_natural_boundary(start: int, end: int) -> int:
        midpoint = start + (chunk_size // 2)
        for boundary in BOUNDARIES:
            pos = text.rfind(boundary, midpoint, end)
            if pos != -1:
                return pos + len(boundary)
        return end
    

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        if end < len(text):
            end = find_natural_boundary(start, end)

        chunks.append(text[start:end])

        if end >= len(text):
            break

        start = max(start + 1, end - overlap)

    return chunks

    

In [52]:


documents = [
    {"text": document_education, "source":"Education"},
    {"text": document_overview, "source":"personal life"},
    {"text": document_professional, "source":"Professional Experiences"}

]

chunks = []
ids = []
metadatas = []


for doc in documents:
    chunks_ = split_text_into_chunks(doc["text"], chunk_size=300, overlap = 30)
    ids_ = [str(uuid.uuid4()) for _ in range(len(chunks_))]
    metadatas_ = [{"source":doc["source"], "chunk_index": i } for i in range(len(chunks_))]

    print(str(ids_))
    chunks.extend(chunks_)
    ids.extend(ids_)
    metadatas.extend(metadatas_)




['e649e20c-0e88-453d-b21a-ed4d8bd62644']
['87d2d42f-035d-4d3e-819b-93eb3ecf7031']
['a6dd7a03-184f-4cd8-9f8b-a04955fed901', '9da6328a-92f5-43da-94f0-9e907b28c047', 'b67f074a-7af9-43fe-b3e6-f276ce74654a', 'd8eff415-2fed-42f6-b308-a793a739fd15', '29f9ffbd-9986-4574-a619-cbe6d0d14e4c', '0ad67919-6189-4bcf-94dc-0f70505d8c59', '706258d2-3dd1-4f8f-b692-81b1a6d3acb4', '6cd31cb0-f155-4057-909d-e876a2c39335', 'b767f860-50fe-4335-a40e-a103ad171a1a', 'e497e231-b496-4657-b2d6-5705f9b10523', '1985c2c4-9c1a-4d81-9fad-6e5a9384b798', 'e5c6cbd7-5a9a-4ee0-a0e3-ec3de333f56f', '7baaffc9-b404-4599-be9a-8ba9e645b1f5', 'df67a379-d92e-4f83-b33c-354ba9b0075f', 'efe91ef6-4b68-4f63-b48f-0765e817edda', '2e365acb-7d40-4f16-92da-ec10e381ea41', '6c7c9b97-be67-4028-ae59-65c21c2190b9', '3a44bedb-13b6-42e6-8254-85d035cdc4cd', '88077019-6413-41ee-a651-65b32684f925', '5ed55c17-0901-4a25-85e7-c7be89d47e76', 'ba530253-d0c7-4e8d-be66-c5e175cf22ce', '63466bb6-fc32-4296-949c-ed3bf445f735', '1321c033-c3a8-4426-9aad-37fc4c9e9992

In [35]:
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} (id: {ids[i]}), Source: {metadatas[i]["source"]}, index: {metadatas[i]["chunk_index"]}) ")
    print(chunk)
    print()

--- Chunk 1 (id: 4acd5013-7c18-416d-a602-b8980551ac80), Source: Education, index: 0) 

In 1999, I Just graduated from Ausbug College

I got my master degree in Software Engineering at University of St. Thomas in 2015
I got my BS at Augsburg College in 1999

--- Chunk 2 (id: 8c92ae03-cd1e-4025-9942-8d16e7ba5192), Source: personal life, index: 0) 

My name is Si Lambda

I am based in Minnesota, United States

Cooking: Enjoy cooking food every day

I love watching soccer and play soccer. I love to watch UEFA Champion Leaguage and FIFA World Cup. 
Since Messi plays for Inter Miami, I watch and follow Messi's team every game.


--- Chunk 3 (id: 0f2ddb79-dfcb-4ab2-b834-4aec5473eeab), Source: Professional Experiences, index: 0) 

I have Microsoft Azure Certified: AI-102, AZ-420, DP-900,DP-700,DP-600,D-100,AZ-400,AZ-104,AZ-204

also I have Amazon Web Service AWS Certified: Administrator and Developer;, 

I am Senior Software Engineer at Red Wing Shoes, working on Front End Javascript, Blazor, 

In [36]:
# Generate embedding
response = client.embeddings.create(
    model = "text-embedding-3-small",
    input = chunks
)

In [37]:
#print(response.data)

embeddings = [item.embedding for item in response.data]
pprint(f"Generated {len(embeddings)} embeddings")
print(f"Each embedding has {len(embeddings[0])} dimension")

'Generated 26 embeddings'
Each embedding has 1536 dimension


In [ ]:

chroma_client = chromadb.PersistentClient(path="./chroma_db_digital_twin")

collection = chroma_client.get_or_create_collection(name="digital_twin")

if collection.get()["ids"]:
    collection.delete(collection.get()["ids"])


#pprint(collection.get())

In [39]:
#prepate data for chroma

collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=chunks,
    metadatas=metadatas
)

#pprint(collection.get(include=["documents"]))

In [40]:
test_query = "Where do Si Lam get Master Degree?"
#test_query2="Team work and collaboration"

response = client.embeddings.create(
    model = "text-embedding-3-small",
    input = [test_query]
)

query_embedding = [response.data[0].embedding]#,response.data[1].embedding]

results = collection.query(
    query_embeddings = query_embedding,
    n_results=3,
    include = ["documents","metadatas"]
)

for chunk, metadata in zip(results["documents"][0], results["metadatas"][0]):
    print(f"Chunk {metadata['chunk_index']} (Source: {metadata['source']}): \n{chunk}\n")

Chunk 0 (Source: Education): 

In 1999, I Just graduated from Ausbug College

I got my master degree in Software Engineering at University of St. Thomas in 2015
I got my BS at Augsburg College in 1999

Chunk 0 (Source: personal life): 

My name is Si Lambda

I am based in Minnesota, United States

Cooking: Enjoy cooking food every day

I love watching soccer and play soccer. I love to watch UEFA Champion Leaguage and FIFA World Cup. 
Since Messi plays for Inter Miami, I watch and follow Messi's team every game.


Chunk 5 (Source: Professional Experiences): 
 Azure DynamoDb, AWS CosmoDB

Contact me at si.lam@serverlessdeveloper.com

Skills with Microsoft Certified: Fabric Analytics Engineer Associate
1-Maintain a data analytics solution
2-Prepare data
3-Implement and manage semantic models






### Step 3: Prepare the list of tools for LLM

In [41]:
tools = []


Step 3a : Add tool calling for pushover

In [ ]:
pushover_user  = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url   = "https://api.pushover.net/1/messages.json"



def send_notification(message: str):
    pay_load = {"user":pushover_user,
                "token": pushover_token,
                "message": message}
    requests.post(pushover_url, data = pay_load)
send_notification_function = {

    "name" : "send_notification",
    "description": "Send a push notification to user phone via pushover. Tell the user important task, event",
    "parameters": {
        "type": "object",
        "properties":{
            "message": {
                "type": "string",
                "description":"The notification message to send to user device"
            }
        },
        "required": ["message"]
    }
}
tools.append({"type": "function","function": send_notification_function})

### Step 3b: Add dice rolling tool 

In [ ]:


# simulate rolling a dice
def dice_roll():
    result = random.randint(1,6)
    return result 

roll_dice_function = {
    "name" : "dice_roll",
    "description": "Return a number by rolling a dice",
    "parameters": {
        "type": "object",
        "properties":{
            
        },
        "required": []
    }
}

tools.append( {"type": "function","function": roll_dice_function})

### Step 4: Function to handle tool calling

In [44]:
def handle_tool_call(tool_calls):
    tools_results = []
    # return what to add to our context (about tool call results, a dictionary)
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        #print(f"Calling function {function_name}")
        args = json.loads(tool_call.function.arguments)

        #Route to function name
        if function_name == "send_notification":

            send_notification(args['message'])
            #print(f"Send notification: {args['message']}")
            content = f"Notitication sent: {args['message']}"
        elif function_name == "dice_roll":

            #print(f"Send notification: {args['message']}")
            content = f"Rolled: {dice_roll()}"
        else:
            content = f"Unknown function {function_name}"

        tool_call_result  = {
            "role":"tool",
            "content":content,
            "tool_call_id":tool_call.id
            
        }

        tools_results.append(tool_call_result)

    return tools_results

### Step 4: Function to Process conversation turn

In [47]:
def response_ai(message, history):
    

    response = client.embeddings.create(
        model = "text-embedding-3-small",
        input = [message]
    )

    query_embedding = response.data[0].embedding

    results = collection.query(
        query_embeddings = query_embedding,
        n_results=3,
        include = ["documents","metadatas"]
    )

    print("\n====================================\n")
    print("**Retrived Chunks**")
    for chunk, metadata in zip(results["documents"][0], results["metadatas"][0]):
        print("----------------------------------------------------------------")
        print(f"<<Document {metadata['source']} -- Chunk {metadata['chunk_index']}\n\n{chunk}\n\n")
    
    system_message_enhanced = system_message + "\n\nContent:\n" + "\n---\n".join(results["documents"][0])
    messages = [{"role":"system", "content": system_message_enhanced}] \
               + history \
               + [{"role":"user", "content": message}] 

    
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages= messages,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message
    loop = 0
    tool_count = len(message.tool_calls) if message is not None and message.tool_calls is not None else 0

    pprint(f"Tools count: {tool_count}")
    while message.tool_calls:
        loop += 1
        if (loop > tool_count): break
        
        #handle tool call
        tool_result = handle_tool_call(message.tool_calls) #whole list of tools

        #add message to context
        messages.append(message)

        #add info about tool call response to context, i.e. mesages
        messages.extend(tool_result)

        #invokde LLM one more time to get its updated response
        response = client.chat.completions.create(
            model='gpt-4.1-mini',
            messages=messages,
            tools=tools #will be in the future
        )
        message = response.choices[0].message
        
    return(message.content)


### Step 6: Call Gradio

In [48]:
gr.ChatInterface(fn=response_ai, autofocus=True, autoscroll=True, submit_btn=True, examples=["hello", "hola", "merhaba"], title="Echo Bot").launch(inbrowser=True) #, share=True)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.




**Retrived Chunks**
----------------------------------------------------------------
<<Document Professional Experiences -- Chunk 20

m.

Show all featured items


Azure Certified: AI-102, AZ-420, DP-900,DP-700,DP-600,D-100,AZ-400,AZ-104,AZ-204
AWS Certified: Admin& Developer;, ReactJS, FastAPI, Serverless AppSenior Software Engineer at Red Wing Shoes.serverlessdeveloper.com


Experience




----------------------------------------------------------------
<<Document Professional Experiences -- Chunk 1

pt, Blazor, and
back end API

You can follow me at my personal website: http://www.serverlessdeveloper.com


My Contact info: silam@hotmail.com 


Here is the list of AWS Certification 

AWS Certified Solution Architect - Associate 
AWS Certified Developer 




----------------------------------------------------------------
<<Document personal life -- Chunk 0


My name is Si Lambda

I am based in Minnesota, United States

Cooking: Enjoy cooking food every day

I love watching soccer a